In [1]:
import pandas as pd
import numpy as np
import mi

In [2]:
reports = [i+4 for i in range(12)]

df_temp = [pd.read_csv(f"reporte ({r}).csv", encoding="latin1") for r in reports]
df = pd.concat(df_temp, ignore_index=True)
df

In [3]:
df_articulos = pd.read_csv('articulos.csv', encoding='latin1')
df_articulos = df_articulos.loc[:, ['Código Artículo','Existencia Período']]
df_articulos

In [4]:
df_cosnolidado = pd.read_csv("consolidado.csv", encoding="latin1")
df_cosnolidado.columns

In [5]:
df_cosnolidado = df_cosnolidado.loc[:, ['contenedor','codigo', 'cantidad', 'costo','cantidad_por_caja', 'CBMM']]
df_cosnolidado.dropna(subset=['codigo', 'CBMM', 'cantidad_por_caja'], inplace=True)
df_cosnolidado['costo'] = pd.to_numeric(
    df_cosnolidado['costo'].astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False).str.strip(),
    errors='coerce'
)
df_cosnolidado.columns


In [6]:
# 1. Separar el texto en listas y expandir a lo largo de las filas
df_expandido = (
    df_cosnolidado.assign(
        codigo=(
            df_cosnolidado['codigo']
            .astype(str)
            .str.strip()
            .str.split(r'\\n|[\n\s]+')
        )
    )
    .explode('codigo')
    .dropna(subset=['codigo'])
)

# 2. Limpieza de espacios residuales en los códigos
df_expandido['codigo'] = df_expandido['codigo'].str.strip()

# 3. Filtrar cadenas vacías resultantes y restablecer índice secuencial
df_expandido = df_expandido[df_expandido['codigo'] != ''].reset_index(
    drop=True
)

# Mostrar las columnas relevantes (ahora cada código ocupa su propia fila)
#df_expandido[['codigo', 'cantidad', 'costo']].head()
df_cosnolidado.dropna(subset=['codigo', 'CBMM', 'cantidad_por_caja'], inplace=True)

df_expandido['codigo'] = df_expandido['codigo'].astype(str).str.strip()
df_expandido

In [7]:
_colss = ['Código Artículo', 'Fecha Transacción', 'Cantidad', 'Total', 'Total Costo']
df_new = df.loc[:, _colss].copy()
df_new['Código Artículo'] = df_new['Código Artículo'].astype(str).str.strip()
df_new['Total'] = pd.to_numeric(df_new['Total'].astype(str).str.strip().str.replace(',', '', regex=False), errors='coerce')
df_new['Total Costo'] = pd.to_numeric(df_new['Total Costo'].astype(str).str.strip().str.replace(',', '', regex=False), errors='coerce')
df_new['Cantidad'] = pd.to_numeric(df_new['Cantidad'], errors='coerce')
df_new.head()


In [8]:
#df_new['Fecha Transacción'] = pd.to_datetime(
#    df_new['Fecha Transacción'],
#    format='%d/%m/%Y %I:%M %p',  # Usa %H:%M si el formato es 24h
#)
# 1. Cast a string, limpieza de espacios/comas y ASIGNACIÓN
df_new['Total'] = (
    df_new['Total']
    .astype(str)
    .str.strip()
    .str.replace(',', '', regex=False)
)
df_new['Total Costo'] = (
    df_new['Total']
    .astype(str)
    .str.strip()
    .str.replace(',', '', regex=False)
)

# 2. Conversión a numérico manejando anomalías
df_new['Total'] = pd.to_numeric(df_new['Total'], errors='coerce')
df_new['Cantidad'] = pd.to_numeric(df_new['Cantidad'], errors='coerce')
ventas_totales = df_new['Total'].sum()
ventas_agrupadas = df_new.groupby('Código Artículo').agg(total_ventas=('Total', 'sum')).sort_values(by='total_ventas', ascending=False)
ventas_agrupadas['Porcentaje'] = ventas_agrupadas['total_ventas'] / ventas_totales
ventas_agrupadas['Porcentaje_acum'] = ventas_agrupadas['Porcentaje'].cumsum()

condiciones = [
    ventas_agrupadas['Porcentaje_acum'] <= .50,
    ventas_agrupadas['Porcentaje_acum'] <= 0.80,  # Zona A
    ventas_agrupadas['Porcentaje_acum'] <= 0.95,  # Zona B
    ventas_agrupadas['Porcentaje_acum'] > 0.95    # Zona C
]
valores = ['AA','A', 'B', 'C']

ventas_agrupadas['Clasificacion'] = np.select(condiciones, valores, default='ND')

ventas_agrupadas[ventas_agrupadas['Clasificacion'] == 'AA']


In [9]:
df_new["Cantidad"] = pd.to_numeric(df_new['Cantidad'], errors='coerce')
df_new.groupby('Código Artículo').agg(
    total_cantidad=('Cantidad', 'sum'),
    promedio_cantidad=('Cantidad', 'mean')
).sort_values(by=['total_cantidad'],ascending=False)

In [10]:
# 1. Asegurar limpieza de la clave en el primer DataFrame
df_new['Código Artículo'] = df_new['Código Artículo'].astype(str).str.strip()

# 2. Seleccionar solo las columnas necesarias del DataFrame maestro/derecho para evitar duplicados
cols_derechas = ['codigo', 'costo', 'cantidad_por_caja', 'CBMM']
maestro = df_expandido[cols_derechas].drop_duplicates(subset=['codigo'])

# 3. Left join
df_resultado = pd.merge(
    df_new,
    maestro,
    how='left',
    left_on='Código Artículo',
    right_on='codigo',
).drop(
    columns=['codigo']
)  # Eliminar la clave redundante
df_new_resultado = pd.merge(
    df_resultado,
    df_articulos,
    how='left',
    left_on='Código Artículo',
    right_on='Código Artículo'
)
# 1. Eliminar comas de miles y recortar espacios
df_new_resultado['Existencia Período'] = (
    df_new_resultado['Existencia Período']
    .astype(str)
    .str.strip()
    .str.replace(',', '', regex=False)
)

# 2. Conversión segura a float
df_new_resultado['Existencia Período'] = pd.to_numeric(
    df_new_resultado['Existencia Período'], errors='coerce'
).fillna(0.0)
df_new_resultado.head(5)

In [11]:
# 1. Asegurar la conversión estricta
# Si el formato original es DD/MM/YYYY con hora de 12h (AM/PM):


# 2. Verificar que la fecha máxima sea consistente (no mayor a agosto)
print('Fecha mínima:', df_new['Fecha Transacción'].min())
print('Fecha máxima:', df_new['Fecha Transacción'].max())

# 3. Regenerar la tabla pivotada
df_values = df_new.pivot_table(
    index='Fecha Transacción',
    columns='Código Artículo',
    values='Cantidad',
    aggfunc='sum',
    fill_value=0,
)

# 4. Volver a graficar
articulos = ['MASS0944', 'MASS1880']
df_values[articulos].plot(kind='line', figsize=(10, 5))

Fecha mínima: 01/02/2024 02:38 PM
Fecha máxima: 31/12/2025 12:57 PM


In [ ]:

# ══════════════════════════════════════════════════════════════════════════════
# MOTOR DE INVENTARIO — POLÍTICAS POR CUADRANTE ADI-CV²
# ══════════════════════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from scipy import stats
import math

PARAMETROS = {
    'flete_cbm': 85.0,
    'capacidad_contenedor_cbm': 68.0,
    'tasa_mantenimiento': 0.15,
    'costo_orden_admin': 50.0,
    'lead_time_semanas': 15,
    'ciclo_revision_semanas': 13,  # ~3 meses, para calcular qty a pedir en Lumpy
    'percentil_ss': 90,            # Percentil empírico para SS en Lumpy/Intermittent
    'adi_umbral': 1.32,
    'cv2_umbral': 0.49,
    'min_pos_ventas': 2,           # Mínimo de semanas con venta positiva para calcular CV²
}
LT = PARAMETROS['lead_time_semanas']

# ────────────────────────────────────────────────────────────────────────────
# 1. CARGA DATOS TRANSACCIONALES
# ────────────────────────────────────────────────────────────────────────────
reports = [i + 4 for i in range(12)]
df_temp = [pd.read_csv(f"reporte ({r}).csv", encoding="latin1", low_memory=False) for r in reports]
df_trans = pd.concat(df_temp, ignore_index=True)

col_art  = [c for c in df_trans.columns if 'digo' in c and 'rt' in c][0]
col_fecha = [c for c in df_trans.columns if 'Fecha' in c][0]

df_trans = df_trans[[col_art, col_fecha, 'Cantidad']].copy()
df_trans.columns = ['sku', 'fecha', 'cantidad']
df_trans['sku'] = df_trans['sku'].astype(str).str.strip()
df_trans['cantidad'] = pd.to_numeric(df_trans['cantidad'], errors='coerce').fillna(0.0)
df_trans['fecha'] = pd.to_datetime(df_trans['fecha'], format='mixed', dayfirst=True)

# ────────────────────────────────────────────────────────────────────────────
# 2. METADATA
# ────────────────────────────────────────────────────────────────────────────
df_art = pd.read_csv('articulos.csv', encoding='latin1')
col_art_art = [c for c in df_art.columns if 'digo' in c and 'rt' in c][0]
col_exist   = [c for c in df_art.columns if 'Existencia' in c][0]
df_art = df_art[[col_art_art, col_exist]].rename(columns={col_art_art: 'sku', col_exist: 'stock_actual'})
df_art['sku'] = df_art['sku'].astype(str).str.strip()
df_art['stock_actual'] = pd.to_numeric(df_art['stock_actual'].astype(str).str.replace(',', ''), errors='coerce').fillna(0.0)
df_art = df_art.drop_duplicates(subset=['sku'], keep='last')

df_cons = pd.read_csv('consolidado.csv', encoding='latin1')
df_cons = df_cons[['codigo', 'costo', 'cantidad_por_caja', 'CBMM']].dropna(subset=['codigo'])
df_cons['costo'] = pd.to_numeric(df_cons['costo'].astype(str).str.replace('$','',regex=False).str.replace(',',''), errors='coerce')
df_cons['cantidad_por_caja'] = pd.to_numeric(df_cons['cantidad_por_caja'], errors='coerce')
df_cons['CBMM'] = pd.to_numeric(df_cons['CBMM'], errors='coerce')
df_cons_exp = (
    df_cons.assign(sku=df_cons['codigo'].astype(str).str.strip().str.split(r'\n|[\n\s]+'))
    .explode('sku').dropna(subset=['sku'])
)
df_cons_exp['sku'] = df_cons_exp['sku'].str.strip()
df_cons_exp = df_cons_exp[df_cons_exp['sku'] != ''].drop_duplicates(subset=['sku'], keep='last')
df_cons_exp = df_cons_exp[['sku', 'costo', 'cantidad_por_caja', 'CBMM']]

# ────────────────────────────────────────────────────────────────────────────
# 3. MATRIZ SEMANAL VECTORIZADA
# ────────────────────────────────────────────────────────────────────────────
df_semanal = df_trans.groupby(
    ['sku', pd.Grouper(key='fecha', freq='W-MON')]
)['cantidad'].sum().unstack(fill_value=0.0)

# Días de historia por SKU (para no truncar nuevos con sanity cap)
history_len = df_trans.groupby('sku')['fecha'].agg(
    first_sale='min', last_sale='max'
)
history_len['weeks_history'] = (history_len['last_sale'] - history_len['first_sale']).dt.days / 7.0

# ────────────────────────────────────────────────────────────────────────────
# 4. CÁLCULO DE ADI, CV², CUADRANTE — VECTORIZADO SKU A SKU
# ────────────────────────────────────────────────────────────────────────────
resultados = []

for sku in df_semanal.index:
    ts = df_semanal.loc[sku]

    # Recortar ceros líderes (antes de la primera venta)
    first_nonzero = ts.ne(0).idxmax() if ts.sum() > 0 else None
    if first_nonzero is None:
        # SKU sin ninguna venta
        resultados.append({'sku': sku, 'cuadrante': 'sin_datos', 'mu_incondicional': 0,
                           'adi': 0, 'cv2': 0, 'n_pos': 0, 'mu_sba': 0,
                           'demanda_esperada_lt': 0, 'ss_empirico': 0, 'rop': 0,
                           'mu_semanal_global': ts.mean(), 'std_semanal_global': ts.std(ddof=1)})
        continue

    ts_trim = ts.loc[first_nonzero:]
    n_periodos = len(ts_trim)
    pos_mask = ts_trim > 0
    pos_sales = ts_trim[pos_mask]
    n_pos = len(pos_sales)

    # ── ADI
    adi = n_periodos / n_pos if n_pos > 0 else float('inf')

    # ── CV²
    if n_pos <= 1:
        cuadrante = 'sin_datos'
        cv2 = np.nan
        mu_pos = pos_sales.iloc[0] if n_pos == 1 else 0
        mu_incondicional = mu_pos / adi if adi > 0 else 0
    else:
        mu_pos = pos_sales.mean()
        std_pos = pos_sales.std(ddof=1)
        cv2 = (std_pos / mu_pos) ** 2 if mu_pos > 0 else 0
        mu_incondicional = ts_trim.mean()   # Equivalente a mu_pos / ADI

        if adi < PARAMETROS['adi_umbral'] and cv2 < PARAMETROS['cv2_umbral']:
            cuadrante = 'smooth'
        elif adi >= PARAMETROS['adi_umbral'] and cv2 < PARAMETROS['cv2_umbral']:
            cuadrante = 'intermittent'
        elif adi < PARAMETROS['adi_umbral'] and cv2 >= PARAMETROS['cv2_umbral']:
            cuadrante = 'erratic'
        else:
            cuadrante = 'lumpy'

    # ── SBA (Syntetos-Boylan Approximation)
    # Aplica corrección de sesgo de Croston: mu_SBA = (mu_pos / ADI) * (1 - CV²/2)
    if cuadrante == 'intermittent' and not np.isnan(cv2):
        mu_sba = mu_incondicional * (1 - cv2 / 2)
        mu_sba = max(0, mu_sba)
    else:
        mu_sba = mu_incondicional  # Lumpy/Smooth/Erratic no aplican corrección SBA

    # ── Demanda Esperada en Lead Time
    demanda_esperada_lt = mu_sba * LT

    # ── Stock de Seguridad Empírico (percentil de errores en ventanas de LT)
    # Ventanas rodantes de LT semanas para calcular la distribución real de errores
    if cuadrante in ('lumpy', 'intermittent') and len(ts_trim) >= LT:
        rolling_demand = ts_trim.rolling(window=LT).sum().dropna()
        errors = rolling_demand - demanda_esperada_lt
        ss_empirico = max(0, np.percentile(errors, PARAMETROS['percentil_ss']))
    elif cuadrante in ('smooth', 'erratic'):
        # Para Smooth/Erratic usamos fórmula Gaussiana clásica
        std_global = ts_trim.std(ddof=1) if len(ts_trim) > 1 else 0
        nivel_srv = 0.95  # Se refinará después con la tabla ABC-XYZ
        z = stats.norm.ppf(nivel_srv)
        ss_empirico = max(0, z * std_global * math.sqrt(LT))
    else:
        ss_empirico = 0

    rop = demanda_esperada_lt + ss_empirico

    resultados.append({
        'sku': sku,
        'cuadrante': cuadrante,
        'adi': round(adi, 3),
        'cv2': round(cv2, 4) if not np.isnan(cv2) else np.nan,
        'n_pos': n_pos,
        'mu_incondicional': mu_incondicional,
        'mu_sba': mu_sba,
        'demanda_esperada_lt': demanda_esperada_lt,
        'ss_empirico': ss_empirico,
        'rop': rop,
        'mu_semanal_global': ts.mean(),
        'std_semanal_global': ts.std(ddof=1),
        'n_semanas_historia': len(ts_trim),
    })

df_sku = pd.DataFrame(resultados)
df_sku = df_sku.merge(history_len[['weeks_history']].reset_index().rename(columns={'sku': 'sku'}), on='sku', how='left')

# ────────────────────────────────────────────────────────────────────────────
# 5. MERGE CON METADATA
# ────────────────────────────────────────────────────────────────────────────
df_sku = df_sku.merge(df_art, on='sku', how='left')
df_sku['stock_actual'] = df_sku['stock_actual'].fillna(0.0)
df_sku = df_sku.merge(df_cons_exp, on='sku', how='left')
df_sku['costo_final'] = df_sku['costo'].fillna(1.0).replace(0.0, np.nan).fillna(1.0)
df_sku['cantidad_por_caja'] = df_sku['cantidad_por_caja'].fillna(1.0).clip(lower=1.0)
df_sku['CBMM'] = df_sku['CBMM'].fillna(0.05).clip(lower=0.001)

# ────────────────────────────────────────────────────────────────────────────
# 6. CLASIFICACIÓN ABC (PARETO)
# ────────────────────────────────────────────────────────────────────────────
ventas_totales_sku = df_trans.groupby('sku')['cantidad'].sum().reset_index().rename(columns={'cantidad': 'total_unidades'})
df_sku = df_sku.merge(ventas_totales_sku, on='sku', how='left')
df_sku['total_unidades'] = df_sku['total_unidades'].fillna(0)

df_sku = df_sku.sort_values('total_unidades', ascending=False).reset_index(drop=True)
tot = df_sku['total_unidades'].sum()
df_sku['pct_acum'] = df_sku['total_unidades'].cumsum() / tot if tot > 0 else 0

df_sku['clase_abc'] = np.select(
    [df_sku['pct_acum'] <= 0.50, df_sku['pct_acum'] <= 0.80, df_sku['pct_acum'] <= 0.95],
    ['AA', 'A', 'B'], default='C'
)

# ────────────────────────────────────────────────────────────────────────────
# 7. NIVEL DE SERVICIO DINÁMICO (TABLA ABC × CUADRANTE)
# ────────────────────────────────────────────────────────────────────────────
NIVEL_SERVICIO_MAP = {
    'AA': 0.97, 'A': 0.95, 'B': 0.90, 'C': 0.85
}
df_sku['nivel_servicio'] = df_sku['clase_abc'].map(NIVEL_SERVICIO_MAP).fillna(0.85)

# Recalcular SS Gaussiano con nivel de servicio correcto para Smooth/Erratic
mask_gauss = df_sku['cuadrante'].isin(['smooth', 'erratic'])
z_scores = stats.norm.ppf(df_sku.loc[mask_gauss, 'nivel_servicio'])
df_sku.loc[mask_gauss, 'ss_empirico'] = np.maximum(
    0,
    z_scores * df_sku.loc[mask_gauss, 'std_semanal_global'] * math.sqrt(LT)
)
df_sku.loc[mask_gauss, 'rop'] = df_sku.loc[mask_gauss, 'demanda_esperada_lt'] + df_sku.loc[mask_gauss, 'ss_empirico']

# ────────────────────────────────────────────────────────────────────────────
# 8. CANTIDAD A PEDIR
# ────────────────────────────────────────────────────────────────────────────
FLETE_CBM = PARAMETROS['flete_cbm']
CICLO = PARAMETROS['ciclo_revision_semanas']

# CBM unitario y costo puesto
df_sku['cbm_unitario'] = df_sku['CBMM'] / df_sku['cantidad_por_caja']
df_sku['costo_flete_unit'] = df_sku['cbm_unitario'] * FLETE_CBM
df_sku['costo_puesto'] = df_sku['costo_final'] + df_sku['costo_flete_unit']
df_sku['h'] = df_sku['costo_puesto'] * PARAMETROS['tasa_mantenimiento']
df_sku['demanda_anual'] = df_sku['mu_incondicional'] * 52.0

# EOQ vectorizado
df_sku['eoq_unidades'] = np.where(
    df_sku['h'] > 0,
    np.sqrt((2 * df_sku['demanda_anual'] * PARAMETROS['costo_orden_admin']) / df_sku['h']),
    0.0
)

# Cobertura de ciclo (para Lumpy/Intermittent, evita usar EOQ con varianza extrema)
df_sku['cobertura_ciclo_unidades'] = df_sku['mu_incondicional'] * CICLO

# Selección de cantidad base por cuadrante
df_sku['qty_base'] = np.select(
    [
        df_sku['cuadrante'].isin(['smooth', 'erratic']),
        df_sku['cuadrante'].isin(['lumpy', 'intermittent']),
    ],
    [df_sku['eoq_unidades'], df_sku['cobertura_ciclo_unidades']],
    default=0.0
)

# Para sin_datos: usar mediana de CBM del catálogo completo como proxy de 1 caja
mediana_cbm_caja = df_cons_exp['CBMM'].median()
qty_sin_datos_default = (mediana_cbm_caja / df_sku.loc[df_sku['cuadrante'] == 'sin_datos', 'CBMM'].replace(0, mediana_cbm_caja)) * df_sku.loc[df_sku['cuadrante'] == 'sin_datos', 'cantidad_por_caja']
df_sku.loc[df_sku['cuadrante'] == 'sin_datos', 'qty_base'] = qty_sin_datos_default.reindex(df_sku[df_sku['cuadrante'] == 'sin_datos'].index).fillna(df_sku.loc[df_sku['cuadrante'] == 'sin_datos', 'cantidad_por_caja'])

# Redondear a múltiplo de caja
df_sku['cajas_a_pedir'] = np.ceil(df_sku['qty_base'] / df_sku['cantidad_por_caja']).clip(lower=1).astype(int)
df_sku['unidades_a_pedir'] = df_sku['cajas_a_pedir'] * df_sku['cantidad_por_caja'].astype(int)
df_sku['cbm_total_pedido'] = df_sku['cajas_a_pedir'] * df_sku['CBMM']
df_sku['inversion_fob_usd'] = df_sku['unidades_a_pedir'] * df_sku['costo_final']

# ────────────────────────────────────────────────────────────────────────────
# 9. DECISIÓN DE REORDEN
# ────────────────────────────────────────────────────────────────────────────
df_sku['posicion_inventario'] = df_sku['stock_actual']
df_sku['requiere_pedido'] = df_sku['posicion_inventario'] <= df_sku['rop']
df_sku['estado'] = np.where(df_sku['requiere_pedido'], 'REORDENAR', 'OK')

for col in ['cajas_a_pedir', 'unidades_a_pedir', 'cbm_total_pedido', 'inversion_fob_usd']:
    df_sku[col] = np.where(df_sku['requiere_pedido'], df_sku[col], 0)

# ────────────────────────────────────────────────────────────────────────────
# 10. EXPORTAR
# ────────────────────────────────────────────────────────────────────────────
columnas_export = [
    'sku', 'cuadrante', 'clase_abc', 'nivel_servicio',
    'stock_actual', 'rop', 'demanda_esperada_lt', 'ss_empirico',
    'mu_incondicional', 'mu_sba', 'adi', 'cv2', 'n_pos', 'n_semanas_historia', 'weeks_history',
    'cajas_a_pedir', 'unidades_a_pedir', 'cbm_total_pedido', 'inversion_fob_usd',
    'costo_final', 'cantidad_por_caja', 'CBMM', 'estado', 'requiere_pedido'
]
df_export = df_sku[[c for c in columnas_export if c in df_sku.columns]].copy()
df_export = df_export.rename(columns={'sku': 'Código Artículo'})
df_export.to_csv('pedidos_requeridos.csv', index=False, encoding='utf-8-sig')

print("═" * 60)
print("MOTOR CUADRANTE — RESUMEN")
print("═" * 60)
print(f"\nDistribución cuadrantes:")
print(df_sku['cuadrante'].value_counts().to_string())
print(f"\nSKUs totales procesados:  {len(df_sku)}")
print(f"SKUs que requieren orden: {df_sku['requiere_pedido'].sum()}")
print(f"Inversión total FOB:      ${df_sku['inversion_fob_usd'].sum():,.0f}")
print(f"CBM total pedidos:        {df_sku['cbm_total_pedido'].sum():,.1f}")
print(f"Contenedores estimados:   {df_sku['cbm_total_pedido'].sum() / 68.0:.1f}")
print(f"\nValidación MASS2221:")
m = df_sku[df_sku['sku'] == 'MASS2221']
if len(m):
    r = m.iloc[0]
    print(f"  Cuadrante: {r['cuadrante']}")
    print(f"  mu_incondicional (sem): {r['mu_incondicional']:.4f}")
    print(f"  Demanda esperada LT: {r['demanda_esperada_lt']:.2f}")
    print(f"  SS empírico P90: {r['ss_empirico']:.2f}")
    print(f"  ROP nuevo: {r['rop']:.2f}")
    print(f"  (Anterior ROP Gaussiano: 6.87)")
else:
    print("  MASS2221 no encontrado")
print("\nArchivo pedidos_requeridos.csv actualizado.")


In [13]:
# ══════════════════════════════════════════════════════════════════════════════
# LOOP PRINCIPAL — CÁLCULO DE INVENTARIO POR SKU
# ══════════════════════════════════════════════════════════════════════════════

PARAMETROS_LOGISTICOS = {
    'flete_cbm': 85.0,
    'capacidad_contenedor_cbm': 68.0,
    'tasa_mantenimiento': 0.15,
    'costo_orden_admin': 50.0,
    'lead_time_semanas': 15,
    'nivel_servicio': 0.90,
}

df_valido = df_new_resultado.dropna(subset=['Cantidad', 'Fecha Transacción']).copy()
df_valido['Fecha Transacción'] = pd.to_datetime(df_valido['Fecha Transacción'], format='mixed', dayfirst=True)
df_valido['Cantidad'] = pd.to_numeric(df_valido['Cantidad'], errors='coerce').fillna(0.0)
if 'Total Costo' in df_valido.columns:
    df_valido['Total Costo'] = pd.to_numeric(df_valido['Total Costo'].astype(str).str.replace(',', ''), errors='coerce')

resultados_lista = []
skus_sin_metadata = []
skus_historial_corto = []
total_skus = df_valido['Código Artículo'].nunique()

for sku, grupo in df_valido.groupby('Código Artículo'):
    grupo = grupo.sort_values('Fecha Transacción')

    costo_fob = grupo['costo'].dropna()
    cbm = grupo['CBMM'].dropna()
    und_caja = grupo['cantidad_por_caja'].dropna()

    if costo_fob.empty:
        costo_fob_val = float(grupo['Total Costo'].mean()) if 'Total Costo' in grupo.columns and not grupo['Total Costo'].dropna().empty else 1.0
        if np.isnan(costo_fob_val) or costo_fob_val <= 0:
            costo_fob_val = 1.0
    else:
        costo_fob_val = float(costo_fob.iloc[-1])

    cbm_val = float(cbm.iloc[-1]) if not cbm.empty else 0.05
    und_caja_val = int(und_caja.iloc[-1]) if not und_caja.empty and und_caja.iloc[-1] > 0 else 1

    col_stock = 'Existencia Período' if 'Existencia Período' in grupo.columns else 'Existencia Periodo'
    stock = float(grupo[col_stock].iloc[-1]) if col_stock in grupo.columns and not grupo[col_stock].dropna().empty else 0.0

    serie_semanal = (
        grupo.set_index('Fecha Transacción')
        .resample('W-MON')['Cantidad']
        .sum()
        .fillna(0.0)
    )

    if len(serie_semanal) < PARAMETROS_LOGISTICOS['lead_time_semanas']:
        skus_historial_corto.append(sku)
        continue

    df_ventas_sku = serie_semanal.to_frame(name='Cantidad')

    metricas = calcular_estadisticas_inventario(
        df_ventas=df_ventas_sku,
        costo_unitario_fob=costo_fob_val,
        cbm_por_caja=cbm_val,
        flete_cbm=PARAMETROS_LOGISTICOS['flete_cbm'],
        capacidad_contenedor_cbm=PARAMETROS_LOGISTICOS['capacidad_contenedor_cbm'],
        tasa_mantenimiento=PARAMETROS_LOGISTICOS['tasa_mantenimiento'],
        costo_orden_admin=PARAMETROS_LOGISTICOS['costo_orden_admin'],
        lead_time_semanas=PARAMETROS_LOGISTICOS['lead_time_semanas'],
        unidades_por_caja=und_caja_val,
        nivel_servicio=PARAMETROS_LOGISTICOS['nivel_servicio'],
        stock_actual=stock,
    )

    detalles = metricas.pop('detalles_adicionales', {})
    metricas['Código Artículo'] = sku
    metricas.update(detalles)
    resultados_lista.append(metricas)

print(f'Total SKUs en df_valido: {total_skus}')
print(f'SKUs con historial corto (< {PARAMETROS_LOGISTICOS["lead_time_semanas"]} sem): {len(skus_historial_corto)}')
print(f'SKUs con resultado final: {len(resultados_lista)}')

df_planificacion = (
    pd.DataFrame(resultados_lista)
    .set_index('Código Artículo')
    .sort_values(by='cbm_total_pedido', ascending=False)
)

print(f'\nMétodos ROP utilizados:')
print(df_planificacion['metodo_rop'].value_counts())


Total SKUs en df_valido: 2048
SKUs con historial corto (< 15 sem): 473
SKUs con resultado final: 1575

Métodos ROP utilizados:
metodo_rop
empirico              1030
parametrico_normal     545
Name: count, dtype: int64


In [14]:
df_planificacion

In [15]:
limite = df_planificacion['demanda_semanal_prom'].quantile(.99)
#limite
d = df_planificacion[df_planificacion['demanda_semanal_prom'] < limite].sort_values(by='demanda_semanal_prom', ascending=False)
d.info()



<class 'pandas.DataFrame'>
Index: 1559 entries, MASS1039 to MASS0976
Data columns (total 19 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   demanda_semanal_prom            1559 non-null   float64
 1   demanda_esperada_lt             1559 non-null   float64
 2   stock_seguridad                 1559 non-null   float64
 3   rop                             1559 non-null   float64
 4   metodo_rop                      1559 non-null   str    
 5   n_ciclos_disponibles            1559 non-null   int64  
 6   posicion_inventario             1559 non-null   float64
 7   requiere_pedido                 1559 non-null   bool   
 8   estado                          1559 non-null   str    
 9   unidades_a_pedir                1559 non-null   int64  
 10  cajas_a_pedir                   1559 non-null   int64  
 11  cbm_total_pedido                1559 non-null   float64
 12  porcentaje_contenedor           1559 no